In [1]:
# Import libraries
import pandas as pd
import os 
import sys

sys.path.append(os.path.abspath(".."))

import functions.wrangling as wrg

# Set working directory
os.chdir(r"G:\.shortcut-targets-by-id\1qO0AfYMqzVbXreDMm-gZUvrYXtVZCnDA\CHL8010F2  CPCSSN Dataset") 

In [4]:
# Load and clean datasets

# Define and load file paths 
file_paths = {
    'patient': 'C4MPatient.csv',
    'lab': 'C4MLab.csv',
    'diag': 'C4MEncounterdiagnosis.csv',
    'condition': 'C4MHealthCondition.csv'
}
datasets = wrg.load_csv(file_paths)

# Specify columns to keep from each dataset
columns_to_keep = {
    'patient': ["Patient_ID", "Sex", "BirthYear"],
    'lab': ["Patient_ID", "Name_calc", "TestResult_calc", "PerformedDate"],
    'diag': ["Patient_ID", "DiagnosisText_calc", "DiagnosisCode_calc", "DateCreated"],
    'condition': ["Patient_ID", "DiagnosisText_calc", "DateCreated"]
}
datasets = wrg.select_columns(datasets, columns_to_keep)

# Clean diagnosis data 
diagnosis_cleaning_steps = [
    ('DiagnosisText_calc', 'uppercase'),
    ('DiagnosisCode_calc', 'strip'),
    ('DiagnosisCode_calc', 'dropna'),
    ('DateCreated', 'datetime')
]
datasets['diag'] = wrg.replace_string_nan(datasets['diag'], 'DiagnosisCode_calc')
datasets['diag'] = wrg.preprocess_data(datasets['diag'], diagnosis_cleaning_steps)

In [6]:
# Extract bipolar disorder lab results and summarize patient counts

# Define BD ICD-9 codes and relevant markers
bd_codes = ["296.0", "296.1", "296.4", "296.5", "296.6", "296.7", "296.80", "296.89"]
relevant_markers = ["TOTAL CHOLESTEROL", "HBA1C", "HDL", "FASTING GLUCOSE", "LDL", "INR", "GLUCOSE TOLERANCE"]

# Extract lab rsesults that occur after BD diagnosis
bd_labs_after = wrg.extract_labs_relative_to_diagnosis(
    lab_df=datasets['lab'],
    diag_df=datasets['diag'],
    diagnosis_codes=bd_codes,
    lab_test_names=relevant_markers
)

# Identify each patient's first BD diagnosis date and code
first_dx_info = datasets['diag'].copy()
first_dx_info['DateCreated'] = pd.to_datetime(first_dx_info['DateCreated'], errors='coerce')

# Sort by patient and diagnosis date, and only keep the first record per patient 
first_dx = first_dx_info.sort_values(['Patient_ID', 'DateCreated']).groupby('Patient_ID').first().reset_index()

# Filter to include only patients whose first diagnosis is BD based on ICD-9 codes
first_dx = first_dx[first_dx['DiagnosisCode_calc'].isin(bd_codes)]

# Rename columns 
first_dx = first_dx.rename(columns={
    'DateCreated': 'BD_Diagnosis_Date',
    'DiagnosisCode_calc': 'BD_Code'
})

# Merge diagnosis date and BD code into lab results dataframe 
bd_labs_after = bd_labs_after.merge(
    first_dx[['Patient_ID', 'BD_Diagnosis_Date', 'BD_Code']], 
    on='Patient_ID', 
    how='left'
)

# Drop 'Lab_Timing' column since it's not needed anymore
bd_labs_after = bd_labs_after.drop(columns=['Lab_Timing'])

# Pivot the lab data to create one row per test, per patient, per day
bd_labs_after = bd_labs_after.pivot_table(
    index=['Patient_ID', 'PerformedDate', 'BD_Diagnosis_Date', 'BD_Code'],
    columns='Name_calc',
    values='TestResult_calc',
    aggfunc='first'
).reset_index()

# Sort final lab dataset by patient and test date
bd_labs_after = bd_labs_after.sort_values(by=['Patient_ID', 'PerformedDate'])

# Classify lab timing (before, after, or both relative to diagnosis)
lab_timing_summary, bd_first_clean = wrg.classify_lab_timing(
    lab_df=datasets['lab'],
    diag_df=datasets['diag'],
    diagnosis_codes=bd_codes
)

# Count how many BD patients had additional non-BD diagnoses on the same day 
non_bd_same_day_count = wrg.other_dx_same_day(
    diag_df=datasets['diag'],
    bd_first_clean=bd_first_clean,
    bd_codes=bd_codes
)

# Output patient counts and lab data availability
summary_stats = [
    ("All patients have BD as their first-ever diagnosis", bd_first_clean['first_any_dx_code'].isin(bd_codes).all()),
    ("Total BD patients (first-ever diagnosis)", bd_first_clean['Patient_ID'].nunique()),
    ("Patients with ANY lab results", lab_timing_summary['Patient_ID'].nunique()),
    ("Patients with labs BEFORE diagnosis", lab_timing_summary[lab_timing_summary['Lab_Data_Timing'] == 'Only before']['Patient_ID'].nunique()),
    ("Patients with labs AFTER diagnosis", lab_timing_summary[lab_timing_summary['Lab_Data_Timing'] == 'Only after']['Patient_ID'].nunique()),
    ("Patients with labs BOTH before and after diagnosis", lab_timing_summary[lab_timing_summary['Lab_Data_Timing'] == 'Both before and after']['Patient_ID'].nunique()),
    ("Number of patients with other diagnoses on the same day", non_bd_same_day_count),
    ("Number of patients with lab results after diagnosis date", bd_labs_after['Patient_ID'].nunique()),
    ("Shape of the lab dataset after filtering", bd_labs_after.shape)
]

for label, value in summary_stats:
    print(f"{label}: {value}")

All patients have BD as their first-ever diagnosis: True
Total BD patients (first-ever diagnosis): 378
Patients with ANY lab results: 220
Patients with labs BEFORE diagnosis: 4
Patients with labs AFTER diagnosis: 195
Patients with labs BOTH before and after diagnosis: 21
Number of patients with other diagnoses on the same day: 105
Number of patients with lab results after diagnosis date: 214
Shape of the lab dataset after filtering: (1196, 11)
